In [ ]:
import re
from pathlib import Path

import pandas as pd

DIR_HASIL = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
FILE_REVIEW = [
    DIR_HASIL / "dataset_review_netral.csv",
    DIR_HASIL / "dataset_review_polar.csv",
]
FILE_MANUAL = [
    Path(r"C:\Users\EZRA\Downloads\Video\label_frasa_2_5_kata.csv"),
    Path(r"C:\Users\EZRA\Downloads\Video\rekomendasi_label_korpus_lanjutan.csv"),
]
DIR_OUT = DIR_HASIL / "hasil_visualisasi"

KOL_KEY = "row_key"
TEXT_COL = "text_clean_bert"
DATE_COL = "tanggal_wib"
KOL_MANUAL, KOL_KONSENSUS, KOL_CATATAN = "label_manual", "label_konsensus", "catatan"
KOL_FINAL, KOL_SUMBER = "label_final", "sumber_label_final"
HANYA_ISI_NETRAL = True

MAP = {
    "positif": "pos",
    "negatif": "neg",
    "netral": "neu",
    "pos": "pos",
    "neg": "neg",
    "neu": "neu",
    "positive": "pos",
    "negative": "neg",
    "neutral": "neu",
}
KELAS = ["neg", "neu", "pos"]


def baca_leksikon(paths):
    bagian = []
    for p in paths:
        if not p.exists():
            raise SystemExit(f"Tidak ditemukan: {p}")
        d = pd.read_csv(p, encoding="utf-8-sig")
        low = {c.strip().lower(): c for c in d.columns}
        k = next((low[c] for c in ("teks", "kata", "frasa", "text") if c in low), None)
        s = next((low[c] for c in low if "status" in c or "label" in c), None)
        if k is None or s is None:
            raise SystemExit(
                f"Kolom teks/status tidak terbaca di {p.name}: {d.columns.tolist()}"
            )
        print(f"  {p.name}: {len(d)} baris -> '{k}' + '{s}'")
        bagian.append(pd.DataFrame({"kata": d[k], "pol": d[s]}))

    x = pd.concat(bagian, ignore_index=True)
    x["kata"] = (
        x["kata"]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    x["pol"] = x["pol"].astype(str).str.lower().str.strip().map(MAP)
    x = x.dropna(subset=["pol"])
    x = x[x["kata"].str.len() > 0].drop_duplicates("kata", keep="last")
    print(f"  entri unik: {len(x)} -> {x['pol'].value_counts().to_dict()}")
    return dict(zip(x["kata"], x["pol"]))


def main():
    bagian = []
    for f in FILE_REVIEW:
        if not f.exists():
            print(f"PERINGATAN: dilewati, tidak ditemukan -> {f.name}")
            continue
        d = pd.read_csv(f)
        d["asal_file"] = f.stem
        print(f"{f.name}: {len(d)} baris")
        bagian.append(d)
    if not bagian:
        raise SystemExit(f"Tidak ada file review yang terbaca di {DIR_HASIL}")

    df = pd.concat(bagian, ignore_index=True, sort=False)
    for c in (TEXT_COL, KOL_KONSENSUS):
        if c not in df.columns:
            raise SystemExit(f"Kolom '{c}' tidak ada: {df.columns.tolist()}")
    for c in (KOL_MANUAL, KOL_CATATAN):
        df[c] = df[c].astype("object") if c in df.columns else pd.NA
    if KOL_KEY in df.columns:
        ganda = int(df[KOL_KEY].duplicated().sum())
        if ganda:
            print(f"PERINGATAN: {ganda} row_key ganda antar file, yang pertama dipakai")
            df = df.drop_duplicates(KOL_KEY, keep="first").reset_index(drop=True)

    n = len(df)
    print(f"Gabungan: {n} baris")
    pol = baca_leksikon(FILE_MANUAL)
    pola = re.compile(
        r"\b(?:"
        + "|".join(re.escape(k) for k in sorted(pol, key=len, reverse=True))
        + r")\b"
    )

    def putuskan(t):
        npos = nneg = 0
        kena = []
        for m in pola.finditer(t):
            p = pol[m.group(0)]
            if p == "pos":
                npos += 1
                kena.append(m.group(0))
            elif p == "neg":
                nneg += 1
                kena.append(m.group(0))
        lab = "neg" if nneg > npos else ("pos" if npos > nneg else None)
        return lab, npos, nneg, ", ".join(dict.fromkeys(kena))

    h = [putuskan(t) for t in df[TEXT_COL].fillna("").astype(str).str.lower()]
    leks = pd.Series([x[0] for x in h], index=df.index)
    df["n_pos_leks"] = [x[1] for x in h]
    df["n_neg_leks"] = [x[2] for x in h]
    df["entri_pemicu"] = [x[3] for x in h]

    tangan = df[KOL_MANUAL].astype(str).str.strip().str.lower().map(MAP)
    kons = df[KOL_KONSENSUS].astype(str).str.strip().str.lower().map(MAP)
    if HANYA_ISI_NETRAL:
        leks = leks.where(kons.eq("neu"))
    dari_leks = tangan.isna() & leks.notna()

    df[KOL_MANUAL] = tangan.fillna(leks)
    df[KOL_FINAL] = df[KOL_MANUAL].fillna(kons)
    df[KOL_SUMBER] = "konsensus"
    df.loc[dari_leks, KOL_SUMBER] = "leksikon"
    df.loc[tangan.notna(), KOL_SUMBER] = "manual_anda"
    kosong = df[KOL_CATATAN].isna() | df[KOL_CATATAN].astype(str).str.strip().isin(
        ["", "nan"]
    )
    df.loc[dari_leks & kosong, KOL_CATATAN] = (
        "leksikon: " + df.loc[dari_leks & kosong, "entri_pemicu"]
    )
    df["label_berubah"] = df[KOL_FINAL].ne(kons)

    sisa = kons.eq("neu") & df[KOL_MANUAL].isna()
    ubah = df["label_berubah"]
    print(
        f"\nmanual_anda {int(tangan.notna().sum())} | leksikon {int(dari_leks.sum())} | "
        f"netral belum diputus {int(sisa.sum())}"
    )
    print(f"Baris berubah label: {int(ubah.sum())} ({ubah.mean():.2%})")
    print(f"\n{'kelas':<6}{'sebelum':>10}{'sesudah':>10}{'selisih':>10}")
    for k in KELAS:
        a, b = int(kons.eq(k).sum()), int(df[KOL_FINAL].eq(k).sum())
        print(f"{k:<6}{a:>10}{b:>10}{b - a:>+10}")
    print(f"\nMatriks perpindahan ({KOL_KONSENSUS} -> {KOL_FINAL}):")
    print(pd.crosstab(kons, df[KOL_FINAL]).to_string())
    print(f"\nAsal label akhir:\n{df[KOL_SUMBER].value_counts().to_string()}")

    d_ubah = df.loc[
        ubah,
        [
            c
            for c in (
                KOL_KEY,
                DATE_COL,
                "asal_file",
                TEXT_COL,
                KOL_KONSENSUS,
                KOL_FINAL,
                KOL_SUMBER,
                "entri_pemicu",
            )
            if c in df.columns
        ],
    ]
    top = (
        d_ubah["entri_pemicu"]
        .dropna()
        .str.split(", ")
        .explode()
        .value_counts()
        .head(25)
    )
    if len(top):
        print(f"\n25 entri pemicu perubahan terbanyak:\n{top.to_string()}")

    nst = pd.DataFrame()
    if DATE_COL in df.columns:
        tgl = pd.to_datetime(df[DATE_COL], errors="coerce")
        d = df.assign(tanggal=tgl.dt.date).dropna(subset=["tanggal"])
        nst = (
            d.groupby("tanggal")[KOL_FINAL]
            .value_counts()
            .unstack(fill_value=0)
            .reset_index()
        )
        nst.columns.name = None
        for k in KELAS:
            if k not in nst.columns:
                nst[k] = 0
        nst["n_total"] = nst[KELAS].sum(axis=1)
        nst["NSt"] = (
            (nst["pos"] - nst["neg"]) / nst["n_total"].where(nst["n_total"] > 0)
        ).round(6)
        print(f"\nNSt harian: {len(nst)} hari, rata-rata {nst['NSt'].mean():.4f}")

    ringkas = pd.DataFrame(
        [
            {
                "n_baris": n,
                "manual_anda": int(tangan.notna().sum()),
                "leksikon": int(dari_leks.sum()),
                "netral_belum_diputus": int(sisa.sum()),
                "berubah": int(ubah.sum()),
                **{f"sebelum_{k}": int(kons.eq(k).sum()) for k in KELAS},
                **{f"sesudah_{k}": int(df[KOL_FINAL].eq(k).sum()) for k in KELAS},
            }
        ]
    )

    kol_sisa = [
        c
        for c in (
            KOL_KEY,
            DATE_COL,
            TEXT_COL,
            KOL_KONSENSUS,
            "n_setuju",
            "conf_konsensus",
            "skor_prioritas",
            KOL_MANUAL,
            KOL_CATATAN,
        )
        if c in df.columns
    ]
    d_sisa = df.loc[sisa, kol_sisa]
    if "skor_prioritas" in kol_sisa:
        d_sisa = d_sisa.sort_values("skor_prioritas", ascending=False)

    DIR_OUT.mkdir(parents=True, exist_ok=True)
    for nama, d in (
        ("dataset_final_gabungan.csv", df),
        ("daftar_perubahan_label.csv", d_ubah),
        ("sisa_review_manual.csv", d_sisa),
        ("nst_harian_final.csv", nst),
        ("ringkasan_gabungan.csv", ringkas),
    ):
        if len(d):
            d.to_csv(DIR_OUT / nama, index=False, encoding="utf-8-sig")
            print(f"Tersimpan: {nama} ({len(d)} baris)")


if __name__ == "__main__":
    main()

dataset_review_netral.csv: 7896 baris
dataset_review_polar.csv: 6154 baris
Gabungan: 14050 baris
  label_frasa_2_5_kata.csv: 786 baris -> 'teks' + 'Status Perubahan'
  rekomendasi_label_korpus_lanjutan.csv: 293 baris -> 'Kata' + 'Status Perubahan'
  entri unik: 962 -> {'pos': 530, 'neg': 390, 'neu': 42}

manual_anda 0 | leksikon 4447 | netral belum diputus 3449
Baris berubah label: 4756 (33.85%)

kelas    sebelum   sesudah   selisih
neg         3733      5398     +1665
neu         7896      3449     -4447
pos         2112      4894     +2782

Matriks perpindahan (label_konsensus -> label_final):
label_final       neg   neu   pos
label_konsensus                  
neg              3733     0     0
neu              1665  3449  2782
pos                 0     0  2112

Asal label akhir:
sumber_label_final
konsensus    9603
leksikon     4447

25 entri pemicu perubahan terbanyak:
entri_pemicu
mencapai               153
                       131
untung                  88
tembus               